# Patch Extraction Analysis

Design analysis for `11_patch_extraction_analysis`, one of the notebooks REPORT.md section 5.6 lists as extension work (`configs/future_extensions/`). The eventual `src/data/crop_patches.py` module does not exist yet; this notebook prototypes the extraction logic against real CBIS-DDSM cases so the module's patch size and sampling ratio are chosen from measured data, not guessed.

The motivation is set out in REPORT.md section 5.5: downsampling a roughly 3000 by 4600 pixel film to 224 by 224 removes the microcalcification signal before training begins. Patch pre-training (Shen et al. [7]) trains on native-resolution lesion crops instead, alongside sampled background crops, before the whole-image stage.

This notebook does not execute here: the DICOM tree lives outside the repository (see `CLAUDE.md`) and is only present on the CUDA host. Every cell below is written to run there against `data/cbis-ddsm`.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom

from src.data.dicom_to_png import _find_dicom
from src.data.manifest import read as read_manifest
from src.data.preprocessing import breast_bbox, breast_mask, load_dicom

RAW_ROOT = Path("../data/cbis-ddsm/cbis_ddsm")
TRAIN_CSV = Path("../data/cbis-ddsm/training/train.csv")

# Native-resolution lesion patch, before any resize to the 224x224 model input.
PATCH_SIZE = 512
N_BACKGROUND_PER_IMAGE = 3
SEED = 42

## Lesion-centred patch extraction

The lesion patch is centred on the ROI mask's centroid rather than its bounding box corner, then clipped so it stays inside the image. Both the source image and the mask are cropped to the same breast bounding box first (`breast_bbox`/`breast_mask`, the same functions `preprocess` uses), so patch coordinates line up with the ROI mask coordinates exactly as `MammogramDataset.load_roi` already assumes for the whole-image pipeline.

In [ ]:
def extract_lesion_patch(
    img: np.ndarray, mask: np.ndarray, patch_size: int = PATCH_SIZE
) -> tuple[np.ndarray, np.ndarray, tuple[int, int, int, int]] | None:
    """Return (image patch, mask patch, (y0, x0, y1, x1)) centred on the lesion.

    None if the mask is empty or the breast-cropped image is smaller than one patch.
    """
    ys, xs = np.where(mask > 0)
    if ys.size == 0:
        return None
    h, w = img.shape
    if h < patch_size or w < patch_size:
        return None
    cy, cx = int(ys.mean()), int(xs.mean())
    y0 = int(np.clip(cy - patch_size // 2, 0, h - patch_size))
    x0 = int(np.clip(cx - patch_size // 2, 0, w - patch_size))
    y1, x1 = y0 + patch_size, x0 + patch_size
    return img[y0:y1, x0:x1], mask[y0:y1, x0:x1], (y0, x0, y1, x1)

## Background patch sampling

Background patches are rejection-sampled: a candidate is accepted only if it does not overlap the lesion box and its centre pixel sits on breast tissue rather than the air background `breast_mask` already segments out. This keeps negative patches representative of tissue the model will actually see, rather than empty film border.

In [ ]:
def sample_background_patches(
    img: np.ndarray,
    tissue_mask: np.ndarray,
    lesion_box: tuple[int, int, int, int],
    patch_size: int,
    n_patches: int,
    rng: np.random.Generator,
    max_tries: int = 200,
) -> list[tuple[int, int]]:
    """Return up to `n_patches` (y0, x0) top-left corners away from the lesion."""
    h, w = img.shape
    ly0, lx0, ly1, lx1 = lesion_box
    patches: list[tuple[int, int]] = []
    tries = 0
    while len(patches) < n_patches and tries < max_tries:
        tries += 1
        y0 = int(rng.integers(0, h - patch_size))
        x0 = int(rng.integers(0, w - patch_size))
        y1, x1 = y0 + patch_size, x0 + patch_size
        overlaps_lesion = y0 < ly1 and y1 > ly0 and x0 < lx1 and x1 > lx0
        if overlaps_lesion:
            continue
        if tissue_mask[y0 + patch_size // 2, x0 + patch_size // 2] == 0:
            continue
        patches.append((y0, x0))
    return patches

## Building a patch manifest over the training split

Mirrors `src/data/cache_roi_masks.py`: only rows with a `roi_mask_id` yield a lesion patch, and the source image and mask DICOMs are located with `_find_dicom` rather than assumed to sit at a fixed path. Every extracted patch, lesion or background, becomes one manifest row so the sampling ratio and patch-size choices below are measured against the actual split rather than assumed.

In [ ]:
rng = np.random.default_rng(SEED)

df = read_manifest(TRAIN_CSV)
has_roi = (
    df["roi_mask_id"].notna() if "roi_mask_id" in df.columns else pd.Series(dtype=bool)
)
lesion_rows = df[has_roi] if len(has_roi) else df.iloc[0:0]
print(f"Training rows: {len(df)}, with an ROI mask: {len(lesion_rows)}")

records = []
skipped = 0
for _, row in lesion_rows.iterrows():
    image_id, rid = str(row["image_id"]), str(row["roi_mask_id"])
    img_dcm = _find_dicom(RAW_ROOT, image_id)
    mask_dcm = _find_dicom(RAW_ROOT, rid)
    if img_dcm is None or mask_dcm is None:
        skipped += 1
        continue

    img = load_dicom(img_dcm)
    tissue = breast_mask(img)
    y0, y1, x0, x1 = breast_bbox(tissue)
    img_c, tissue_c = img[y0:y1, x0:x1], tissue[y0:y1, x0:x1]

    mask = (pydicom.dcmread(str(mask_dcm)).pixel_array > 0).astype(np.uint8)
    if mask.shape != img.shape:
        mask = cv2.resize(
            mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST
        )
    mask_c = mask[y0:y1, x0:x1]

    lesion = extract_lesion_patch(img_c, mask_c, PATCH_SIZE)
    if lesion is None:
        skipped += 1
        continue
    _, lesion_mask_patch, lesion_box = lesion
    records.append(
        {
            "image_id": image_id,
            "roi_mask_id": rid,
            "patch_type": "lesion",
            "lesion_area_px": int((mask_c > 0).sum()),
            "lesion_bbox_diag": float(
                np.hypot(*[b1 - b0 for b0, b1 in zip(lesion_box[:2], lesion_box[2:])])
            ),
            "patch_lesion_frac": float(lesion_mask_patch.mean()),
        }
    )

    for by0, bx0 in sample_background_patches(
        img_c, tissue_c, lesion_box, PATCH_SIZE, N_BACKGROUND_PER_IMAGE, rng
    ):
        records.append(
            {
                "image_id": image_id,
                "roi_mask_id": rid,
                "patch_type": "background",
                "lesion_area_px": 0,
                "lesion_bbox_diag": float("nan"),
                "patch_lesion_frac": 0.0,
            }
        )

patch_df = pd.DataFrame.from_records(records)
print(f"Skipped (missing DICOM or image smaller than one patch): {skipped}")
print(f"Patches extracted: {len(patch_df)}")
patch_df["patch_type"].value_counts()

## Sampling ratio and patch-size diagnostics

Two numbers the eventual `crop_patches.py` needs as arguments: the lesion bounding-box size distribution, which sets `PATCH_SIZE` (too small crops the lesion, too large wastes background-patch diversity), and the realised lesion-to-background ratio, which is `N_BACKGROUND_PER_IMAGE` scaled by how many training rows actually carry an ROI.

In [ ]:
lesion_diag = patch_df.loc[patch_df["patch_type"] == "lesion", "lesion_bbox_diag"]
print("Lesion bounding-box diagonal (pixels), native resolution:")
print(lesion_diag.describe())
print()
print(
    f"PATCH_SIZE={PATCH_SIZE} covers "
    f"{(lesion_diag <= PATCH_SIZE).mean() * 100:.1f}% of lesions without clipping the ROI."
)

n_lesion = int((patch_df["patch_type"] == "lesion").sum())
n_background = int((patch_df["patch_type"] == "background").sum())
print(
    f"\nRealised sampling ratio, lesion:background = 1:{n_background / n_lesion:.2f}"
    f" ({n_lesion} lesion patches, {n_background} background patches)"
)

fig, ax = plt.subplots(figsize=(6, 3))
lesion_diag.plot(kind="hist", bins=30, ax=ax)
ax.axvline(PATCH_SIZE, color="red", linestyle="--", label=f"PATCH_SIZE={PATCH_SIZE}")
ax.set_xlabel("Lesion bounding-box diagonal (px)")
ax.set_ylabel("Count")
ax.set_title("Native-resolution lesion size distribution")
ax.legend()
plt.tight_layout()
plt.show()

## Visual sanity check

One training case, native resolution, breast-cropped: the lesion patch and one sampled background patch drawn as boxes over the full breast-cropped image. This is the check to run first on the CUDA host, before trusting the manifest above: if the red box does not sit on the ROI contour, the bounding-box math has a coordinate bug.

In [ ]:
row = lesion_rows.iloc[0]
image_id, rid = str(row["image_id"]), str(row["roi_mask_id"])
img_dcm = _find_dicom(RAW_ROOT, image_id)
mask_dcm = _find_dicom(RAW_ROOT, rid)

img = load_dicom(img_dcm)
tissue = breast_mask(img)
y0, y1, x0, x1 = breast_bbox(tissue)
img_c = img[y0:y1, x0:x1]

mask = (pydicom.dcmread(str(mask_dcm)).pixel_array > 0).astype(np.uint8)
if mask.shape != img.shape:
    mask = cv2.resize(
        mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST
    )
mask_c = mask[y0:y1, x0:x1]

lesion = extract_lesion_patch(img_c, mask_c, PATCH_SIZE)
_, _, (ly0, lx0, ly1, lx1) = lesion
bg_boxes = sample_background_patches(
    img_c,
    tissue[y0:y1, x0:x1],
    (ly0, lx0, ly1, lx1),
    PATCH_SIZE,
    1,
    np.random.default_rng(SEED),
)

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(img_c, cmap="gray")
ax.contour(mask_c, colors="lime", linewidths=1)
ax.add_patch(
    mpatches.Rectangle(
        (lx0, ly0),
        PATCH_SIZE,
        PATCH_SIZE,
        edgecolor="red",
        facecolor="none",
        linewidth=2,
        label="lesion patch",
    )
)
for by0, bx0 in bg_boxes:
    ax.add_patch(
        mpatches.Rectangle(
            (bx0, by0),
            PATCH_SIZE,
            PATCH_SIZE,
            edgecolor="cyan",
            facecolor="none",
            linewidth=2,
            label="background patch",
        )
    )
ax.set_title(f"{image_id} (ROI in green, lesion patch red, background patch cyan)")
ax.axis("off")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Notes for `src/data/crop_patches.py`

- `PATCH_SIZE` should be set from the diagonal distribution above, not fixed at 512 a priori; re-run this notebook once the CUDA host is reachable and read off the percentile that keeps clipped lesions rare.
- The lesion:background ratio realised here (`N_BACKGROUND_PER_IMAGE` per lesion row) only covers rows with an ROI. Mass and calcification training rows without one contribute no lesion patch and should either be dropped from patch pre-training or sampled purely for background, a decision this notebook does not make.
- `extract_lesion_patch` and `sample_background_patches` are copied as-is into the module signature `crop_patches.py` would need; the manifest-building cell above is the loop `crop_patches.main()` would run, writing patches to disk instead of collecting them in `patch_df`.